Pablo Méndez, Rodrigo Dueñas, Roberto Howe and Jordi Casaldàliga

### 1. Importing Libraries

We’ll start by importing the necessary Python libraries.  
We mainly use **pandas** for data manipulation and **os** for handling file paths.

In [1]:
import pandas as pd
import os
from pathlib import Path
from datetime import datetime
from tabulate import tabulate

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import argparse

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

### 2. Defining the Data Paths

Here we specify the paths to the two folders that contain our CSV files.  
The idea is to automatically go through each file in the main archive and check whether a matching file exists in the secondary one.

In [2]:
secondary_archive_path  = '../data/mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16'
main_archive_path = '../data/mturkfitbit_export_4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16'
full_path = '../data/full_path'
if not os.path.exists(full_path):
	os.makedirs(full_path)

### 3. Listing Files

We now list all the `.csv` files in the main folder.  
This will help us iterate automatically over them and perform the merge operation for each.

In [3]:
main_files = [f for f in os.listdir(main_archive_path) if f.endswith('.csv')]
main_files

['dailyActivity_merged.csv',
 'dailyCalories_merged.csv',
 'dailyIntensities_merged.csv',
 'dailySteps_merged.csv',
 'heartrate_seconds_merged.csv',
 'hourlyCalories_merged.csv',
 'hourlyIntensities_merged.csv',
 'hourlySteps_merged.csv',
 'minuteCaloriesNarrow_merged.csv',
 'minuteCaloriesWide_merged.csv',
 'minuteIntensitiesNarrow_merged.csv',
 'minuteIntensitiesWide_merged.csv',
 'minuteMETsNarrow_merged.csv',
 'minuteSleep_merged.csv',
 'minuteStepsNarrow_merged.csv',
 'minuteStepsWide_merged.csv',
 'sleepDay_merged.csv',
 'weightLogInfo_merged.csv']

### 4. Merging the Files

For each CSV file in the main archive:
1. We read it into a DataFrame.
2. We check if there’s a corresponding file with the same name in the secondary archive.
3. If it exists, we concatenate both DataFrames.
4. We remove duplicates (to avoid repeated records).
5. We save the merged DataFrame as a new CSV, with another name.

In [4]:
for file in main_files:
	# Read the CSV file from the main archive
	main_df = pd.read_csv(os.path.join(main_archive_path, file))
	secondary_file_path = os.path.join(secondary_archive_path, file)

	if os.path.exists(secondary_file_path):
		# Read the corresponding CSV file from the secondary archive
		secondary_df = pd.read_csv(secondary_file_path)

		# Merge the two dataframes
		merged_df = pd.concat([main_df, secondary_df], ignore_index=True)

		#remove duplicates if needed
		merged_df = merged_df.drop_duplicates()

		# Save the merged dataframe
		merged_df.to_csv(os.path.join(full_path, f'full_{file}'), index=False)
		print(f"Merged {file} with its secondary counterpart.")
	else:
		main_df.to_csv(os.path.join(full_path, f'full_{file}'), index=False)
		print(f"No secondary file found for {file}. Skipping merge.")

Merged dailyActivity_merged.csv with its secondary counterpart.
No secondary file found for dailyCalories_merged.csv. Skipping merge.
No secondary file found for dailyIntensities_merged.csv. Skipping merge.
No secondary file found for dailySteps_merged.csv. Skipping merge.
Merged heartrate_seconds_merged.csv with its secondary counterpart.
Merged hourlyCalories_merged.csv with its secondary counterpart.
Merged hourlyIntensities_merged.csv with its secondary counterpart.
Merged hourlySteps_merged.csv with its secondary counterpart.
Merged minuteCaloriesNarrow_merged.csv with its secondary counterpart.
No secondary file found for minuteCaloriesWide_merged.csv. Skipping merge.
Merged minuteIntensitiesNarrow_merged.csv with its secondary counterpart.
No secondary file found for minuteIntensitiesWide_merged.csv. Skipping merge.
Merged minuteMETsNarrow_merged.csv with its secondary counterpart.
Merged minuteSleep_merged.csv with its secondary counterpart.
Merged minuteStepsNarrow_merged.csv 

### 5. Helper Functions

To make our code modular and readable, we’ll define several helper functions that handle small, specific tasks such as formatting dates, numbers, and summarizing columns.

---

##### `format_date(date_str)`

This function tries to interpret strings as dates.  
Some Fitbit files store timestamps in different formats (with or without hours), so we check both.

---

##### `format_number(value)`

This function ensures numeric outputs (like averages or standard deviations)  
are printed with **three decimal places**.

---

##### `process_col(column, serie)`

This function calculates **descriptive statistics** for one column:
- Count of non-null values  
- Number of missing values  
- Number of distinct values  
- (If numeric) → min, max, mean, and standard deviation  

It returns a dictionary containing these statistics, ready to be turned into a table.

---

##### `log_row(text, fp)`

A simple utility that writes text to both:
- the **console** (so we see progress live),
- and a **log file** (to keep results saved).


In [5]:
def format_date(date_str):
	ret = pd.NaT
	try:
		ret = datetime.strptime(date_str, '%m/%d/%Y %I:%M:%S %p')
	except ValueError:
		pass

	try:
		ret = datetime.strptime(date_str, '%m/%d/%Y')
	except ValueError:
		pass

	return ret

def format_number(value):
	if isinstance(value, (int, float)):
		return f"{value:.3f}"
	return value

def process_col(column, serie):
	row = {'Column': column}

	row.update({
		'Count': format_number(serie.count()),
		'Null Count': format_number(serie.isnull().sum()),
		'Distinct Count': format_number(serie.nunique())
	})

	if pd.api.types.is_numeric_dtype(serie):
	  row.update({
		  'Min': format_number(serie.min()),
		  'Max': format_number(serie.max()),
		  'Average': format_number(serie.mean()),
		  'Std Dev': format_number(serie.std())
	  })

	return row

def log_row(text, fp):
  print(text)
  fp.write(text + "\n")

## 6. Clean the data

Now that we have helper functions, we can define the main data-cleaning function.

### What it does:
1. Opens a report file (`table_statistics.txt`).
2. Loops through every `.csv` file in the dataset folder.
3. Loads the file into a DataFrame.
4. Detects and converts columns that look like dates.
5. Computes per-column statistics using `process_col()`.
6. Logs those statistics as a formatted table using `tabulate`.
7. Saves a cleaned version of the CSV into a new folder.

In [6]:
def main(datadir, cleandir):
	with open('table_statistics.txt', 'w') as f:
		for filename in os.listdir(datadir):
			if not filename.endswith('.csv'): continue

			file_path = os.path.join(datadir, filename)
			df = pd.read_csv(file_path)

			log_row(f"Statistics for {filename}:", f)

			stats = []
			date_columns_modified = False

			for column in df.columns:
				if df[column].dtype == 'object' and df[column].apply(lambda x: isinstance(x, str)).all():
					try:
						date_column = df[column].apply(format_date)
						if not date_column.isnull().all():
							df[column] = date_column
					except ValueError:
						pass

				row = process_col(column, df[column])
				stats.append(row)

			log_row(tabulate(stats, headers="keys", tablefmt="grid"), f)

			new_file_path = os.path.join(cleandir, f"{filename}")
			df.to_csv(new_file_path, index=False)

			f.flush()

if __name__ == "__main__":
		data ="../data/full_path"
		clean = '../data/full_path_clean'

		if not os.path.exists(clean):
			os.makedirs(clean)

		main(data, clean)

Statistics for full_dailyActivity_merged.csv:
+--------------------------+---------+--------------+------------------+-------------+-----------------+----------------+----------------+
| Column                   |   Count |   Null Count |   Distinct Count |         Min |             Max |        Average |        Std Dev |
+==========================+=========+==============+==================+=============+=================+================+================+
| Id                       |    1397 |            0 |               35 | 1.50396e+09 |     8.87769e+09 |    4.78121e+09 |    2.38429e+09 |
+--------------------------+---------+--------------+------------------+-------------+-----------------+----------------+----------------+
| ActivityDate             |    1397 |            0 |               62 |             |                 |                |                |
+--------------------------+---------+--------------+------------------+-------------+-----------------+----------------


## 7. Creating a Unified DataFrame

We combine the cleaned CSV files into a single DataFrame using the variables specified in the dataset description.

Since the data is recorded at different temporal resolutions, we use a **daily resolution**, with one row per user and day.

- **Daily activity:** Steps, distances, activity durations, sedentary time, and calories.
- **Sleep:** Total minutes asleep and total time in bed.
- **Weight:** Weight in kilograms and BMI.
- **METs:** Minute-level measurements aggregated into daily averages.

The resulting `unified_df` contains the relevant variables for further analysis. Missing values are preserved when measurements are unavailable, and the original high-resolution data remains accessible in the corresponding CSV files.


In [7]:
data_path = "../data/full_path_clean"

# Daily activity
unified_df = pd.read_csv(os.path.join(data_path, "full_dailyActivity_merged.csv"))

unified_df["Date"] = pd.to_datetime(unified_df["ActivityDate"]).dt.normalize()

unified_df = unified_df.drop(columns=["ActivityDate", "TrackerDistance", "LoggedActivitiesDistance", "SedentaryActiveDistance"])

unified_df = unified_df.drop_duplicates(subset=["Id", "Date"], keep="first")
# Sleep
sleep_df = pd.read_csv(os.path.join(data_path, "full_sleepDay_merged.csv"))

sleep_df["Date"] = pd.to_datetime(sleep_df["SleepDay"]).dt.normalize()

sleep_daily = sleep_df.groupby(["Id", "Date"], as_index=False)[
	["TotalMinutesAsleep", "TotalTimeInBed"]
	].sum()

unified_df = unified_df.merge(sleep_daily, on=["Id", "Date"], how="left")

# Weight and BMI
weight_df = pd.read_csv(os.path.join(data_path, "full_weightLogInfo_merged.csv"))

weight_df["Date"] = pd.to_datetime(weight_df["Date"])
weight_df = weight_df.sort_values("Date")
weight_df["Date"] = weight_df["Date"].dt.normalize()

weight_df = weight_df.drop_duplicates(subset=["Id", "Date"], keep="last")

unified_df = unified_df.merge(weight_df[["Id", "Date", "WeightKg", "BMI"]], on=["Id", "Date"], how="left")

# Daily average METs
mets_df = pd.read_csv(os.path.join(data_path, "full_minuteMETsNarrow_merged.csv"))

mets_df["Date"] = pd.to_datetime(mets_df["ActivityMinute"]).dt.normalize()
mets_df["METs"] = mets_df["METs"] / 10

mets_daily = (mets_df.groupby(["Id", "Date"], as_index=False).agg(METs_mean=("METs", "mean")))

unified_df = unified_df.merge(mets_daily, on=["Id", "Date"], how="left")

# Sort and save
unified_df = unified_df.sort_values(["Id", "Date"]).reset_index(drop=True)

unified_df.to_csv(
	os.path.join(data_path, "unified_df.csv"),
	index=False
)

print("Shape:", unified_df.shape)
display(unified_df.head())

C:\Users\Pablo\AppData\Local\Temp\ipykernel_18648\591598569.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sleep_df["Date"] = pd.to_datetime(sleep_df["SleepDay"]).dt.normalize()
C:\Users\Pablo\AppData\Local\Temp\ipykernel_18648\591598569.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  weight_df["Date"] = pd.to_datetime(weight_df["Date"])
C:\Users\Pablo\AppData\Local\Temp\ipykernel_18648\591598569.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  mets_df["Date"] = pd.to_datetime(mets_df["ActivityMinute"]).dt.normalize()


Shape: (1373, 17)


,Id,TotalSteps,TotalDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories,Date,TotalMinutesAsleep,TotalTimeInBed,WeightKg,BMI,METs_mean
0,1503960366,11004,7.11,2.57,0.46,4.07,33,12,205,804,1819,2016-03-25,NaN,NaN,NaN,NaN,1.601806
1,1503960366,17609,11.55,6.92,0.73,3.91,89,17,274,588,2154,2016-03-26,NaN,NaN,NaN,NaN,1.895903
2,1503960366,12736,8.53,4.66,0.16,3.71,56,5,268,605,1944,2016-03-27,NaN,NaN,NaN,NaN,1.711458
3,1503960366,13231,8.93,3.19,0.79,4.95,39,20,224,1080,1932,2016-03-28,NaN,NaN,NaN,NaN,1.701250
4,1503960366,12041,7.85,2.16,1.09,4.61,28,28,243,763,1886,2016-03-29,NaN,NaN,NaN,NaN,1.660694


In [8]:
print(unified_df.isna().sum())

Id                             0
TotalSteps                     0
TotalDistance                  0
VeryActiveDistance             0
ModeratelyActiveDistance       0
LightActiveDistance            0
VeryActiveMinutes              0
FairlyActiveMinutes            0
LightlyActiveMinutes           0
SedentaryMinutes               0
Calories                       0
Date                           0
TotalMinutesAsleep           963
TotalTimeInBed               963
WeightKg                    1275
BMI                         1275
METs_mean                     31
dtype: int64


#T1

Completed functions** --> Revisarlo

Full code for the exercises --> pendiente de terminar a partir de predict_sleep

In [9]:
# Duplicated Function from the cell above
def prepare_sleep_prediction_data(activity_df, sleep_df):
	"""
	Prepare features and target for sleep prediction
	activity of day D -> minutes asleep in the night after D (the sleep that ends on D + 1)
	"""
	activity_df = activity_df.copy()
	sleep_df = sleep_df.copy()

	# Dates as real dates (in the cleaned CSVs they are text such as '2016-04-12')
	activity_df['ActivityDate'] = pd.to_datetime(activity_df['ActivityDate'])
	sleep_df['SleepDay'] = pd.to_datetime(sleep_df['SleepDay'])

	# SleepDay is the date on which the sleep ENDS, so the night after day D has SleepDay = D + 1
	activity_df['NextSleepDay'] = activity_df['ActivityDate'] + pd.to_timedelta(1, unit='D')

	# Merge activity and sleep data
	df = activity_df.merge(sleep_df, left_on=['Id', 'NextSleepDay'], right_on=['Id', 'SleepDay'], how='inner')

	# Select features to use
	features = [
		'TotalSteps', 'TotalDistance',
		'VeryActiveDistance', 'ModeratelyActiveDistance', 'LightActiveDistance',
		'VeryActiveMinutes', 'FairlyActiveMinutes', 'LightlyActiveMinutes'
	]

	# Select target to predict
	target = ['TotalMinutesAsleep']

	# Split features and target
	X = df[features].values
	y = df[target].values

	return X, y

def predict_sleep(model, scaler, features):
	"""
	Make predictions using the trained model
	"""
	# Scale features
	scaled_features = scaler.transform(features)

	# Convert to tensor and predict
	model.eval()
	with torch.no_grad():
		inputs = torch.tensor(scaled_features, dtype=torch.float32)
		prediction = model(inputs).numpy()

	return prediction

class FitbitSleepPredictor(nn.Module):
	def __init__(self, input_size):
		"""
		Initialize the FitbitSleepPredictor neural network.

		Args:
			input_size (int): The number of input features.
		"""
		super(FitbitSleepPredictor, self).__init__()

		# Define the neural network architecture using a Sequential container
		self.network = nn.Sequential(
			nn.Linear(input_size, 64),
			nn.ReLU(),
			nn.Dropout(0.2),

			nn.Linear(64, 32),
			nn.ReLU(),
			nn.Dropout(0.2),

			nn.Linear(32, 16),
			nn.ReLU(),

			nn.Linear(16, 1)
		)

	def forward(self, x):
		"""
		Defines the forward pass of the neural network.

		Args:
			x (torch.Tensor): Input tensor containing the features.

		Returns:
			torch.Tensor: The predicted sleep duration.
		"""
		# Pass the input through the entire network and return the result
		return self.network(x)


def train_sleep_predictor(X, y, epochs=300, batch_size=32, learning_rate=0.001):
	"""
	Train a neural network model to predict sleep duration based on Fitbit activity data.

	Args:
		X (numpy.ndarray): Input features (activity data)
		y (numpy.ndarray): Target values (sleep duration)
		epochs (int): Number of training epochs
		batch_size (int): Size of mini-batches for training

	Returns:
		tuple: Trained model and feature scaler
	"""
	# Split the data into training and testing sets
	# 80% for training, 20% for testing
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

	# Initialize and fit the StandardScaler to normalize the feature data
	scaler = StandardScaler()
	
	# Fit the scaler to the training data and transform it
	X_train = scaler.fit_transform(X_train)

	# Transform the test data using the fitted scaler
	X_test = scaler.transform(X_test)

	# Convert numpy arrays to PyTorch tensors for use with PyTorch models
	X_train = torch.tensor(X_train, dtype=torch.float32)
	X_test = torch.tensor(X_test, dtype=torch.float32)

	y_train = torch.tensor(y_train, dtype=torch.float32)
	y_test = torch.tensor(y_test, dtype=torch.float32)

	# Create an instance of the FitbitSleepPredictor model
	# The input size is set to the number of features in the training data
	model = FitbitSleepPredictor(input_size=X_train.shape[1])

	# Define the loss function (Mean Squared Error for regression tasks)
	criterion = nn.MSELoss()

	# Define the optimizer (Adam) with a learning rate of 0.001
	optimizer = optim.Adam(model.parameters(), lr=learning_rate)

	# Initialize lists to store training and validation losses
	train_losses = []
	val_losses = []

	# Training loop
	for epoch in range(epochs):
		# Set the model to training mode (enables dropout, etc.)
		model.train()
		epoch_train_loss = 0

		# Mini-batch training

		for i in range(0, len(X_train), batch_size):
			# Extract a mini-batch of data
			batch_X = X_train[i:i+batch_size]
			batch_y = y_train[i:i+batch_size]

			# Forward pass: compute the model output for this batch
			outputs = model(batch_X)

			# Compute the loss between the predicted and actual values
			loss = criterion(outputs, batch_y)

			# Backward pass and optimization
			# Clear previous gradients
			# This is necessary because PyTorch accumulates gradients by default.
			# If we don't clear them, the gradients from the previous batch would be
			# added to the current batch's gradients, leading to incorrect updates.
			# By zeroing the gradients, we ensure that each batch starts with a clean slate.
			optimizer.zero_grad()

			loss.backward()        # Compute gradients
			optimizer.step()       # Update model parameters

			epoch_train_loss += loss.item()

		# Calculate average training loss for the epoch
		avg_train_loss = epoch_train_loss / (len(X_train) / batch_size)
		train_losses.append(avg_train_loss)

		# Validation
		model.eval()
		with torch.no_grad():
			val_predictions = model(X_test)
			val_loss = criterion(val_predictions, y_test)
			val_losses.append(val_loss.item())

		if epoch % 10 == 0:
			# Print the current epoch and test loss
			print(f'Epoch [{epoch}/{epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss.item():.4f}')

	# Plot training and validation loss
	plt.figure(figsize=(10, 5))
	plt.plot(train_losses, label='Training Loss')
	plt.plot(val_losses, label='Validation Loss')
	plt.xlabel('Epoch')
	plt.ylabel('Loss')
	plt.title('Training and Validation Loss')
	plt.legend()
	plt.savefig('loss_plot.png')
	plt.close()

	# Return the trained model and the scaler for later use in predictions
	return model, scaler

def evaluate_model(model, X_test, y_test):
	model.eval()

	with torch.no_grad():
		predictions = model(X_test)

		mse = nn.MSELoss()(predictions, y_test)
		mae = nn.L1Loss()(predictions, y_test)

		rmse = torch.sqrt(mse)

	print(f"RMSE: {rmse.item():.2f} minutes")
	print(f"MAE: {mae.item():.2f} minutes")

	return rmse.item(), mae.item()


if __name__ == "__main__":

	# Set up argument parser
	parser = argparse.ArgumentParser(description='Train and evaluate a sleep prediction model')
	parser.add_argument('--datadir', type=str, default='../data/full_path_clean', help='Directory containing the data files')
	parser.add_argument('--epochs', type=int, default=300, help='Number of training epochs')
	parser.add_argument('--batch_size', type=int, default=32, help='Size of mini-batches for training')
	parser.add_argument('--learning_rate', type=float, default=0.001, help='Learning rate for the optimizer')
	parser.add_argument('--test_size', type=float, default=0.2, help='Proportion of data to use for testing')
	parser.add_argument('--random_state', type=int, default=42, help='Random state for reproducibility')
	args = parser.parse_args(args=[])

	# Load your data
	activity_df = pd.read_csv(os.path.join(args.datadir, 'full_dailyActivity_merged.csv'))
	sleep_df = pd.read_csv(os.path.join(args.datadir, 'full_sleepDay_merged.csv'))	

	# Prepare data
	X, y = prepare_sleep_prediction_data(activity_df, sleep_df)	

	# Train model
	model, scaler = train_sleep_predictor(X, y, epochs=args.epochs, batch_size=args.batch_size, learning_rate=args.learning_rate)	

	# Evaluate model
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=args.test_size, random_state=args.random_state)

	X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
	y_test = torch.tensor(y_test, dtype=torch.float32)
	evaluate_model(model, X_test, y_test)

	# Make predictions
	new_features = np.array([[10000, 8.0, 2.0, 3.0, 3.0, 30, 45, 180]])  # Example input
	predicted_sleep = predict_sleep(model, scaler, new_features)
	print(f"Predicted sleep duration: {predicted_sleep[0][0]:.2f} minutes")

C:\Users\Pablo\AppData\Local\Temp\ipykernel_18648\965721271.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sleep_df['SleepDay'] = pd.to_datetime(sleep_df['SleepDay'])


Epoch [0/300], Train Loss: 201864.0706, Val Loss: 173421.8594
Epoch [10/300], Train Loss: 173756.7559, Val Loss: 141074.6406
Epoch [20/300], Train Loss: 42899.4165, Val Loss: 28702.3672
Epoch [30/300], Train Loss: 33005.4173, Val Loss: 21956.8711
Epoch [40/300], Train Loss: 29214.1507, Val Loss: 19301.9258
Epoch [50/300], Train Loss: 25620.7772, Val Loss: 17332.4258
Epoch [60/300], Train Loss: 22406.6849, Val Loss: 15953.0469
Epoch [70/300], Train Loss: 19693.7878, Val Loss: 15257.9912
Epoch [80/300], Train Loss: 18017.3907, Val Loss: 14729.8516
Epoch [90/300], Train Loss: 16820.4410, Val Loss: 14576.6709
Epoch [100/300], Train Loss: 16786.4171, Val Loss: 14413.0342
Epoch [110/300], Train Loss: 16240.3842, Val Loss: 14597.3027
Epoch [120/300], Train Loss: 16856.4844, Val Loss: 14361.9355
Epoch [130/300], Train Loss: 16699.5824, Val Loss: 14751.1045
Epoch [140/300], Train Loss: 17897.4594, Val Loss: 14373.1426
Epoch [150/300], Train Loss: 15251.2899, Val Loss: 14556.1592
Epoch [160/300]